In [ ]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 25.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# ============================================================
# CELL 1 – Imports
# ============================================================
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn.functional as F

In [ ]:
# ============================================================
# CELL 2 – Load data (fixed with cast_column)
# ============================================================
from datasets import load_dataset, concatenate_datasets, Value

def load_data(sample_size=15000):
    amazon = load_dataset("amazon_polarity")
    train_amazon = amazon['train'].shuffle(seed=42).select(range(sample_size))
    test_amazon  = amazon['test'].shuffle(seed=42).select(range(2000))

    yelp = load_dataset("yelp_polarity")
    train_yelp = yelp['train'].shuffle(seed=42).select(range(5000))
    test_yelp  = yelp['test'].shuffle(seed=42).select(range(1000))

    # Rename yelp 'text' → 'content'
    train_yelp = train_yelp.rename_column("text", "content")
    test_yelp  = test_yelp.rename_column("text",  "content")

    # Keep only the columns we need
    cols = ['content', 'label']
    train_amazon = train_amazon.select_columns(cols)
    test_amazon  = test_amazon.select_columns(cols)
    train_yelp   = train_yelp.select_columns(cols)
    test_yelp    = test_yelp.select_columns(cols)

    # ✅ FIX: force the Arrow schema of label to plain int64 in ALL datasets
    for ds in [train_amazon, test_amazon, train_yelp, test_yelp]:
        ds = ds  # reference; cast_column returns a new dataset

    train_amazon = train_amazon.cast_column("label", Value("int64"))
    test_amazon  = test_amazon.cast_column("label",  Value("int64"))
    train_yelp   = train_yelp.cast_column("label",   Value("int64"))
    test_yelp    = test_yelp.cast_column("label",    Value("int64"))

    train_dataset = concatenate_datasets([train_amazon, train_yelp]).shuffle(seed=42)
    test_dataset  = concatenate_datasets([test_amazon,  test_yelp]).shuffle(seed=42)

    return train_dataset, test_dataset

In [ ]:
# ============================================================
# CELL 3 – Tokenize (increase max_length for longer context)
# ============================================================
def tokenize_data(dataset, tokenizer, max_length=256):
    """
    Longer max_length captures more context, important for
    sarcastic sentences where the twist comes later.
    """
    def tokenize_function(example):
        return tokenizer(
            example['content'],
            padding='max_length',
            truncation=True,
            max_length=max_length
        )
    return dataset.map(tokenize_function, batched=True)

In [ ]:
# ============================================================
# CELL 4 – Model (unchanged)
# ============================================================
def load_model():
    model = RobertaForSequenceClassification.from_pretrained(
        'roberta-base',
        num_labels=2
    )
    return model

In [ ]:
# ============================================================
# CELL 5 – Metrics (unchanged)
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1": f1,
            "precision": precision, "recall": recall}

In [ ]:
# ============================================================
# CELL 6 – Training args (more epochs, warmup, scheduler)
# ============================================================
def get_training_args():
    return TrainingArguments(
        output_dir="./results",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,          # was 2
        weight_decay=0.01,
        warmup_ratio=0.1,            # NEW: warmup prevents early overfitting
        lr_scheduler_type="cosine",  # NEW: cosine decay
        eval_strategy="epoch",       # NEW: evaluate every epoch
        save_strategy="epoch",
        load_best_model_at_end=True, # NEW: keeps the best checkpoint
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),  # mixed precision on GPU
    )

In [ ]:
# ============================================================
# CELL 7 – Train (unchanged)
# ============================================================
def train_model(model, train_dataset, test_dataset, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    trainer.train()
    return trainer

In [ ]:
# ============================================================
# CELL 8 – Improved predict_sentiment with confidence score
# ============================================================
def predict_sentiment(text, model, tokenizer, max_length=256, threshold=0.65):
    """
    Returns label AND confidence. If confidence < threshold,
    the model flags the text as 'Uncertain (possible sarcasm)'.

    Args:
        threshold: minimum softmax probability to commit to a label.
                   Sarcastic sentences often produce ~0.55 confidence.
    """
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)[0]   # [neg_prob, pos_prob]
    pred  = torch.argmax(probs).item()
    conf  = probs[pred].item()

    label = "Positive" if pred == 1 else "Negative"

    if conf < threshold:
        return f"Uncertain / Possibly Sarcastic ({label} @ {conf:.0%} confidence)"
    return f"{label} ({conf:.0%} confident)"

In [ ]:
# ============================================================
# CELL 9 – Main
# ============================================================
def main():
    train_data, test_data = load_data()

    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

    train_data = tokenize_data(train_data, tokenizer)
    test_data  = tokenize_data(test_data,  tokenizer)

    model        = load_model()
    training_args = get_training_args()
    trainer      = train_model(model, train_data, test_data, training_args)

    results = trainer.evaluate()
    print("\nEvaluation Results:", results)

    model.save_pretrained("sentiment_model")
    tokenizer.save_pretrained("sentiment_model")

if __name__ == "__main__":
    main()

Casting the dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Evaluation Results: {'eval_loss': 0.24086181819438934, 'eval_accuracy': 0.9586666666666667, 'eval_f1': 0.9586666666666667, 'eval_precision': 0.9599465954606141, 'eval_recall': 0.9573901464713716, 'eval_runtime': 11.9149, 'eval_samples_per_second': 251.786, 'eval_steps_per_second': 31.473, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL 10 – Inference loop
# ============================================================
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

model     = RobertaForSequenceClassification.from_pretrained("sentiment_model")
tokenizer = RobertaTokenizer.from_pretrained("sentiment_model")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

while True:
    user_input = input("\nEnter review (or type 'exit'): ")
    if user_input.lower() == "exit":
        break
    result = predict_sentiment(user_input, model, tokenizer)
    print("Sentiment:", result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: sentiment_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`